In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras import backend as K

print("TensorFlow version:", tf.__version__)

2026-05-09 14:37:51.482698: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-09 14:37:51.544525: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-09 14:37:51.619449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-09 14:37:51.698082: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-09 14:37:51.698379: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-09 14:37:51.805030: I tensorflow/core/platform/cpu_feature_guard.cc:

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Simple but good enough CNN for CIFAR-10
def build_cifar_model():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Dropout(0.25),
        
        Conv2D(64, (3,3), activation='relu', padding='same'),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Dropout(0.25),
        
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cifar_model = build_cifar_model()
cifar_model.fit(x_train, y_train, epochs=15, batch_size=64, validation_data=(x_test, y_test), verbose=1)

/home/saidul/Desktop/4-2/Deep Learning/myenv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15


2026-05-09 14:38:01.642971: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 614400000 exceeds 10% of free system memory.


782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.3173 - loss: 1.8349

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = Model([model.inputs], 
                       [model.get_layer(last_conv_layer_name).output, model.output])
    
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    heatmap = np.maximum(heatmap, 0) / np.max(heatmap) if np.max(heatmap) != 0 else heatmap
    return heatmap

# Last conv layer name (check model.summary())
last_conv_layer = "conv2d_3"   # Change according to your model.summary()

In [ ]:
def show_explainability(model, img, true_label, last_conv_layer_name, technique="Grad-CAM"):
    img_array = np.expand_dims(img, axis=0)
    preds = model.predict(img_array, verbose=0)
    pred_class = np.argmax(preds[0])
    
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f"Original\nTrue: {true_label}, Pred: {pred_class}")
    plt.axis('off')
    
    if technique == "Grad-CAM":
        heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=pred_class)
        heatmap = cv2.resize(heatmap.numpy(), (img.shape[1], img.shape[0]))
        heatmap = np.uint8(255 * heatmap)
        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        superimposed = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)
        
        plt.subplot(1, 3, 2)
        plt.imshow(heatmap)
        plt.title("Grad-CAM Heatmap")
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(superimposed)
        plt.title("Superimposed")
        plt.axis('off')
    plt.show()

In [ ]:
cifar_class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                     'dog', 'frog', 'horse', 'ship', 'truck']

# Show explanations for 3 test images
for i in range(3):
    show_explainability(cifar_model, x_test[i], 
                       cifar_class_names[np.argmax(y_test[i])],
                       last_conv_layer_name=last_conv_layer)

In [ ]:
# Load your face dataset (same structure as previous assignments)
# Assume you have folder: ../data/faces/ with subfolders person_0, person_1, ..., person_6

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    '/home/saidul/Desktop/4-2/Deep Learning/data/faces',
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training')

val_generator = train_datagen.flow_from_directory(
    '/home/saidul/Desktop/4-2/Deep Learning/data/faces',
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation')

# Build simple face classifier
face_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

face_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
face_model.fit(train_generator, epochs=10, validation_data=val_generator)

In [ ]:
# Test on one of your face images
test_img_path = '../data/faces/person_0/angle_0.jpg'   # Change path
test_img = tf.keras.preprocessing.image.load_img(test_img_path, target_size=(64,64))
test_array = tf.keras.preprocessing.image.img_to_array(test_img) / 255.0

show_explainability(face_model, test_array, 
                   f"Person {np.argmax(face_model.predict(np.expand_dims(test_array,0), verbose=0))}",
                   last_conv_layer_name="conv2d_8")   # Update layer name after face_model.summary()